In [2]:
# pip install earthengine-api

In [6]:
# earthengine authenticate
# need to run this only once
# to send tasks to google earth engine, we need a googl cloud proejct with access to earth engine api
# the access requires approval from google as of September 2025
# academic researh project are in the scope of approval
ee.Authenticate()

True

### new s1 pipeline

In [ ]:
import ee, json, time

GEOJSON_PATH       = '../dubai_regions.geojson'
PROJECT_ID         = 'earth-444815'
START_DATE         = '2015-03-23'
END_DATE           = '2025-09-30'
SLEEP_BETWEEN_TASK = 2

ee.Initialize(project=PROJECT_ID)

def weekly_dates(start, end):
    s = ee.Date(start)
    e = ee.Date(end)
    n_weeks = e.difference(s, 'week').subtract(1)
    return ee.List.sequence(0, n_weeks).map(lambda k: s.advance(k, 'week'))

with open(GEOJSON_PATH) as f:
    gj = json.load(f)

features = [ee.Feature(ee.Geometry(feat['geometry']),
                       {'name': feat['properties']['name']}) for feat in gj['features']]
region_fc = ee.FeatureCollection(features)

s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterDate(START_DATE, END_DATE)
      .filterBounds(region_fc.geometry())
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')))

week_starts = weekly_dates(START_DATE, END_DATE)

for region_feat in features:
    reg_name = region_feat.get('name').getInfo()
    reg_geom = region_feat.geometry()

    def region_series(date):
        d_start = ee.Date(date)
        d_end   = d_start.advance(1, 'week')
        weekly  = s1.filterDate(d_start, d_end)
        comp    = weekly.median()
        count   = weekly.size()

        vv_sel = ee.Image(ee.Algorithms.If(
            count.gt(0),
            comp.select('VV'),
            ee.Image.constant(0).rename('VV').updateMask(ee.Image(0))
        ))
        vh_sel = ee.Image(ee.Algorithms.If(
            count.gt(0),
            comp.select('VH'),
            ee.Image.constant(0).rename('VH').updateMask(ee.Image(0))
        ))

        ten = ee.Image.constant(10)
        vv_lin = ten.pow(vv_sel.divide(10))
        vh_lin = ten.pow(vh_sel.divide(10))
        ratio_lin = vv_lin.divide(vh_lin).rename('RATIO_LIN')
        ratio_db  = ratio_lin.log10().multiply(10).rename('RATIO_DB')

        reducer = (ee.Reducer.mean()
                   .combine(ee.Reducer.median(), sharedInputs=True)
                   .combine(ee.Reducer.percentile([90]), sharedInputs=True))

        stats_vv = vv_sel.reduceRegion(reducer=reducer, geometry=reg_geom, scale=10, bestEffort=True)
        stats_vh = vh_sel.reduceRegion(reducer=reducer, geometry=reg_geom, scale=10, bestEffort=True)
        stats_ratio_lin = ratio_lin.reduceRegion(reducer=ee.Reducer.mean(), geometry=reg_geom, scale=10, bestEffort=True)
        stats_ratio_db  = ratio_db.reduceRegion(reducer=ee.Reducer.mean(), geometry=reg_geom, scale=10, bestEffort=True)

        vv_dict   = ee.Dictionary(stats_vv)
        vh_dict   = ee.Dictionary(stats_vh)
        rlin_dict = ee.Dictionary(stats_ratio_lin)
        rdb_dict  = ee.Dictionary(stats_ratio_db)

        vv_mean    = ee.Algorithms.If(vv_dict.contains('VV_mean'), vv_dict.get('VV_mean'), None)
        vv_median  = ee.Algorithms.If(vv_dict.contains('VV_median'), vv_dict.get('VV_median'), None)
        vv_p90     = ee.Algorithms.If(vv_dict.contains('VV_p90'), vv_dict.get('VV_p90'), None)
        vh_mean    = ee.Algorithms.If(vh_dict.contains('VH_mean'), vh_dict.get('VH_mean'), None)
        vh_median  = ee.Algorithms.If(vh_dict.contains('VH_median'), vh_dict.get('VH_median'), None)
        vh_p90     = ee.Algorithms.If(vh_dict.contains('VH_p90'), vh_dict.get('VH_p90'), None)
        ratio_lin_mean = ee.Algorithms.If(rlin_dict.contains('RATIO_LIN'), rlin_dict.get('RATIO_LIN'), None)
        ratio_db_mean  = ee.Algorithms.If(rdb_dict.contains('RATIO_DB'),  rdb_dict.get('RATIO_DB'),  None)

        return ee.Feature(None, {
            'region':               reg_name,
            'week_start':           d_start.format('YYYY-MM-dd'),
            'vv_mean_db':           vv_mean,
            'vv_median_db':         vv_median,
            'vv_p90_db':            vv_p90,
            'vh_mean_db':           vh_mean,
            'vh_median_db':         vh_median,
            'vh_p90_db':            vh_p90,
            'vv_vh_ratio_mean_lin': ratio_lin_mean,
            'vv_vh_ratio_mean_db':  ratio_db_mean,
            'n_obs':                count
        })

    series_fc = ee.FeatureCollection(week_starts.map(region_series))

    ee.batch.Export.table.toDrive(
        collection     = series_fc,
        description    = f'{reg_name}_weekly_s1sar',
        fileNamePrefix = f'{reg_name}_weekly_s1sar',
        fileFormat     = 'CSV'
    ).start()

    time.sleep(SLEEP_BETWEEN_TASK)

print('All export tasks submitted.')


All export tasks submitted.
